In [1]:
from pathlib import Path
import pandas as pd

NB_DIR = Path('.').resolve()  # avaliacao_chats_comerciais/
DATA_DIR = NB_DIR / 'data'
DF_AVAL_PATH = NB_DIR.parent / 'avaliacao_llms' / 'df_avaliacoes.xlsx'
BATCHES_DIR = NB_DIR / 'batches'
BATCHES_DIR.mkdir(exist_ok=True)

def prepara_batches(concurso: str, chats: dict, n_batches: int = 5):
    """
    concurso: 'bndes' | 'cvm' | 'petrobras'
    chats: {'chatgpt': Path, 'claude_chat': Path} com xlsx contendo colunas id, resposta
    """
    df = pd.read_excel(DF_AVAL_PATH)
    sub = df[df['edital'] == concurso][['id', 'categoria', 'pergunta', 'modelo', 'resposta']]

    wide = (sub.pivot_table(index=['id', 'categoria', 'pergunta'],
                            columns='modelo', values='resposta', aggfunc='first')
              .reset_index())
    wide.columns = list(wide.columns[:3]) + [f'resposta_{c}' for c in wide.columns[3:]]

    for nome, path in chats.items():
        c = pd.read_excel(path)[['id', 'resposta']].rename(columns={'resposta': f'resposta_{nome}'})
        wide = wide.merge(c, on='id', how='left')

    wide = wide.sort_values('id').reset_index(drop=True)
    assert len(wide) == 50, f'esperado 50 perguntas, veio {len(wide)}'

    size = len(wide) // n_batches
    for i in range(n_batches):
        ini, fim = i * size, (i + 1) * size if i < n_batches - 1 else len(wide)
        out = BATCHES_DIR / f'{concurso}_batch_{i+1:02d}.xlsx'
        wide.iloc[ini:fim].to_excel(out, index=False)
        print(f'{out.name}: perguntas {wide.iloc[ini].id}–{wide.iloc[fim-1].id}')
    return wide

prepara_batches('bndes', {
    'chatgpt':     DATA_DIR / 'bndes_std_chatgpt_qa.xlsx',
    'claude_chat': DATA_DIR / 'bndes_std_claude_qa.xlsx',
})

bndes_batch_01.xlsx: perguntas 1–10
bndes_batch_02.xlsx: perguntas 11–20
bndes_batch_03.xlsx: perguntas 21–30
bndes_batch_04.xlsx: perguntas 31–40
bndes_batch_05.xlsx: perguntas 41–50


,id,categoria,pergunta,resposta_claude-haiku-4-5,resposta_claude-opus-4-7,resposta_claude-sonnet-4-6,resposta_deepseek-v4-flash,resposta_deepseek-v4-pro,resposta_gpt-4o-mini,resposta_gpt-5.4,resposta_gpt-5.4-mini,resposta_gpt-5.5,resposta_chatgpt,resposta_claude_chat
0,1,Dados sobre as inscrições,Qual o período de inscrição para o concurso BN...,**Período de inscrição do BNDES (01/2024):**\n...,O período de inscrição para o concurso BNDES (...,Aqui estão as informações sobre o período de i...,O período de inscrição para o concurso BNDES (...,O período de inscrição para o concurso do BNDE...,O período de inscrição para o concurso do BNDE...,O período de inscrição foi de 26/07/2024 a 19/...,O período de inscrição foi de 26/07/2024 a 19/...,Hoje é 30/04/2026. O período de inscrição para...,O período de inscrição para a Seleção Pública ...,O período de inscrições foi de\n26 de julho a ...
1,2,Dados sobre as inscrições,Qual o valor da taxa de inscrição do concurso ...,A **taxa de inscrição do concurso BNDES é de R...,A taxa de inscrição do concurso BNDES (Edital ...,O valor da taxa de inscrição do concurso BNDES...,A taxa de inscrição para o concurso do **BNDES...,O valor da taxa de inscrição do concurso BNDES...,O valor da taxa de inscrição para o concurso d...,"A taxa de inscrição é de R$ 110,00.\n\nref: ed...","A taxa de inscrição é de R$ 110,00.\n\nSe você...",A taxa de inscrição do concurso BNDES é de R$ ...,O valor da taxa de inscrição do concurso do\nB...,O valor da taxa de inscrição do concurso BNDES...
2,3,Dados sobre as inscrições,Em qual site o candidato deve fazer a inscriçã...,O candidato deve fazer a inscrição no **site d...,A inscrição deve ser feita pelo site da **Fund...,A inscrição para o concurso do BNDES deve ser ...,A inscrição para o concurso do BNDES (Edital n...,O edital menciona que a inscrição deve ser fei...,Os candidatos devem realizar a inscrição para ...,A inscrição deve ser feita na página da Fundaç...,O candidato deve fazer a inscrição na página d...,A inscrição deve ser feita somente pela intern...,A inscrição do concurso do\nBanco Nacional de ...,A inscrição do concurso BNDES deveria ser feit...
3,4,Dados sobre as inscrições,Quais são as formas de pagamento da taxa de in...,"Com base no edital do BNDES, as formas de paga...",O edital do BNDES (01/2024) prevê **duas forma...,O edital do BNDES (01/2024) prevê **duas forma...,De acordo com o edital do concurso BNDES (01/2...,"De acordo com o edital, após o envio do requer...",As formas de pagamento da taxa de inscrição do...,As formas de pagamento da taxa de inscrição sã...,As formas de pagamento da taxa de inscrição sã...,As formas de pagamento da taxa de inscrição sã...,De acordo com o edital do concurso do\nBanco N...,"De acordo com o edital, havia\nduas formas de ..."
4,5,Dados sobre as inscrições,Qual é o prazo-limite de pagamento da inscriçã...,"Com base no edital do BNDES, o **prazo-limite ...",O edital não fixa explicitamente uma data únic...,O prazo-limite para pagamento da inscrição no ...,"De acordo com o edital, o prazo-limite de paga...","Com base no edital, o prazo para pagamento da ...",O prazo-limite para o pagamento da inscrição d...,O edital informa que o pagamento da inscrição ...,O edital informa que o pagamento da inscrição ...,"Hoje é 30/04/2026. No edital, as inscrições fo...",O edital informa que o pagamento da inscrição ...,O edital\nnão especifica uma data-limite única...
5,6,Dados sobre as inscrições,Quem pode pedir isenção da taxa de inscrição d...,"Com base no edital do BNDES, **podem pedir ise...",## Quem pode pedir isenção da taxa de inscriçã...,Ótima pergunta! Veja quem tem direito à isençã...,Podem pedir **isenção total** da taxa de inscr...,Com base no edital do concurso BNDES (01/2024)...,Os candidatos que podem solicitar a isenção da...,Pode pedir isenção da taxa de inscrição quem s...,Podem pedir isenção da taxa de inscrição os(as...,Pode pedir isenção total da taxa de inscrição ...,Podiam solicitar isenção da taxa

In [3]:
from pathlib import Path
import pandas as pd

NB_DIR = Path('.').resolve()  # avaliacao_chats_comerciais/
DATA_DIR = NB_DIR / 'data'
DF_AVAL_PATH = NB_DIR.parent / 'avaliacao_llms' / 'df_avaliacoes.xlsx'
BATCHES_DIR = NB_DIR / 'batches'
BATCHES_DIR.mkdir(exist_ok=True)

def consolida(concurso: str, n_batches: int = 5) -> pd.DataFrame:
    EVAL_DIR = NB_DIR / 'batches_eval'
    partes = [pd.read_excel(EVAL_DIR / f'{concurso}_batch_{i+1:02d}_eval.xlsx')
              for i in range(n_batches)]
    final = pd.concat(partes, ignore_index=True)

    esperado = {'id', 'pergunta', 'modelo', 'resposta', 'score', 'justificativa'}
    faltando = esperado - set(final.columns)
    assert not faltando, f'colunas faltando: {faltando}'
    assert len(final) == 550, f'esperado 550 linhas, veio {len(final)}'
    assert final['score'].isin([0, 0.5, 1]).all(), 'scores fora do domínio'

    final.to_excel(NB_DIR / f'{concurso}_eval_final.xlsx', index=False)

    # placar rápido por modelo
    placar = (final.groupby('modelo')['score']
                   .agg(['mean', 'sum', 'count'])
                   .sort_values('mean', ascending=False))
    print(placar)
    return final

bndes_final = consolida('bndes')
bndes_final

                   mean   sum  count
modelo                              
gpt-5.5            0.94  47.0     50
gpt-5.4            0.92  46.0     50
gpt-5.4-mini       0.84  42.0     50
gpt-4o-mini        0.77  38.5     50
deepseek-v4-pro    0.77  38.5     50
claude-opus-4-7    0.77  38.5     50
claude-haiku-4-5   0.72  36.0     50
deepseek-v4-flash  0.69  34.5     50
chatgpt            0.66  33.0     50
claude-sonnet-4-6  0.66  33.0     50
claude_chat        0.49  24.5     50


,id,pergunta,modelo,resposta,score,justificativa
0,1,Qual o período de inscrição para o concurso BN...,claude-haiku-4-5,**Período de inscrição do BNDES (01/2024):**\n...,0.5,item 6.2.1 fixa inscrição de 10h de 26/07/2024...
1,1,Qual o período de inscrição para o concurso BN...,claude-opus-4-7,O período de inscrição para o concurso BNDES (...,0.5,item 6.2.1 fixa inscrição de 10h de 26/07/2024...
2,1,Qual o período de inscrição para o concurso BN...,claude-sonnet-4-6,Aqui estão as informações sobre o período de i...,0.5,item 6.2.1 fixa inscrição de 10h de 26/07/2024...
3,1,Qual o período de inscrição para o concurso BN...,deepseek-v4-flash,O período de inscrição para o concurso BNDES (...,0.5,item 6.2.1 fixa inscrição de 10h de 26/07/2024...
4,1,Qual o período de inscrição para o concurso BN...,deepseek-v4-pro,O período de inscrição para o concurso do BNDE...,0.5,item 6.2.1 fixa inscrição de 10h de 26/07/2024...
...,...,...,...,...,...,...
545,50,Quais cuidados o candidato deve ter com o cart...,gpt-5.4,Os principais cuidados com o Cartão-Resposta s...,1.0,itens 9.16 a 9.18.1 exigem caneta preta transp...
546,50,Quais cuidados o candidato deve ter com o cart...,gpt-5.4-mini,O candidato deve:\n\n- usar caneta esferográfi...,1.0,itens 9.16 a 9.18.1 exigem caneta preta transp...
547,50,Quais cuidados o candidato deve ter com o cart...,gpt-5.5,O candidato deve tomar estes cuidados com o Ca...,1.0,itens 9.16 a 9.18.1 exigem caneta preta transp...
548,50,Quais cuidados o candidato deve ter com o cart...,chatgpt,No concurso do\nBanco Nacional de Desenvolvime...,0.5,itens 9.16 a 9.18.1 tratam dos cuidados com ma...


In [7]:
from pathlib import Path
import pandas as pd

NB_DIR = Path('.').resolve()  # avaliacao_chats_comerciais/
DATA_DIR = NB_DIR / 'data'
DF_AVAL_PATH = NB_DIR.parent / 'avaliacao_llms' / 'df_avaliacoes.xlsx'
BATCHES_DIR = NB_DIR / 'batches'
BATCHES_DIR.mkdir(exist_ok=True)

def prepara_batches(concurso: str, chats: dict, n_batches: int = 5):
    """
    concurso: 'bndes' | 'cvm' | 'petrobras'
    chats: {'chatgpt': Path, 'claude_chat': Path} com xlsx contendo colunas id, resposta
    """
    df = pd.read_excel(DF_AVAL_PATH)
    sub = df[df['edital'] == concurso][['id', 'categoria', 'pergunta', 'modelo', 'resposta']]

    wide = (sub.pivot_table(index=['id', 'categoria', 'pergunta'],
                            columns='modelo', values='resposta', aggfunc='first')
              .reset_index())
    wide.columns = list(wide.columns[:3]) + [f'resposta_{c}' for c in wide.columns[3:]]

    for nome, path in chats.items():
        c = pd.read_excel(path)[['id', 'resposta']].rename(columns={'resposta': f'resposta_{nome}'})
        wide = wide.merge(c, on='id', how='left')

    wide = wide.sort_values('id').reset_index(drop=True)
    assert len(wide) == 50, f'esperado 50 perguntas, veio {len(wide)}'

    size = len(wide) // n_batches
    for i in range(n_batches):
        ini, fim = i * size, (i + 1) * size if i < n_batches - 1 else len(wide)
        out = BATCHES_DIR / f'{concurso}_batch_{i+1:02d}.xlsx'
        wide.iloc[ini:fim].to_excel(out, index=False)
        print(f'{out.name}: perguntas {wide.iloc[ini].id}–{wide.iloc[fim-1].id}')
    return wide

prepara_batches('petrobras', {
    'chatgpt':     DATA_DIR / 'petro_std_chatgpt_qa.xlsx',
    'claude_chat': DATA_DIR / 'petro_std_claude_qa.xlsx',
})

petrobras_batch_01.xlsx: perguntas 1–10
petrobras_batch_02.xlsx: perguntas 11–20
petrobras_batch_03.xlsx: perguntas 21–30
petrobras_batch_04.xlsx: perguntas 31–40
petrobras_batch_05.xlsx: perguntas 41–50


,id,categoria,pergunta,resposta_claude-haiku-4-5,resposta_claude-opus-4-7,resposta_claude-sonnet-4-6,resposta_deepseek-v4-flash,resposta_deepseek-v4-pro,resposta_gpt-4o-mini,resposta_gpt-5.4,resposta_gpt-5.4-mini,resposta_gpt-5.5,resposta_chatgpt,resposta_claude_chat
0,1,Dados sobre as inscrições,Qual o período de inscrição para o concurso PE...,"Com base no edital, o **período de inscrição f...",**Período de inscrição:** de **17/12/2021 a 05...,Aqui estão as informações sobre o período de i...,O período de inscrição para o concurso da **Pe...,O período de inscrição para o concurso da Petr...,O período de inscrição para o concurso da Petr...,O período de inscrição foi de 17/12/2021 a 05/...,Hoje é 29/04/2026. O edital informa que a insc...,Hoje é 30/04/2026. A inscrição para o concurso...,O período de inscrição para o Processo Seletiv...,De acordo com o Anexo IV (Cronograma) do edita...
1,2,Dados sobre as inscrições,Qual o valor da taxa de inscrição do concurso ...,A **taxa de inscrição do concurso Petrobras é ...,A taxa de inscrição do concurso da Petrobras (...,O valor da taxa de inscrição do concurso da Pe...,A taxa de inscrição do concurso **Petrobras — ...,A taxa de inscrição do concurso da Petrobras (...,O valor da taxa de inscrição para o concurso d...,"A taxa de inscrição é de R$ 79,83.\n\nref: edi...","A taxa de inscrição é de **R$ 79,83**.\n\nref:...",A taxa de inscrição do concurso Petrobras é de...,O valor da taxa de inscrição do concurso PETRO...,"De acordo com o item\n5.1\ndo edital, o valor ..."
2,3,Dados sobre as inscrições,Em qual site o candidato deve fazer a inscriçã...,O candidato deve fazer a inscrição no site do ...,A inscrição deve ser feita exclusivamente pela...,O candidato deve realizar a inscrição **exclus...,A inscrição para o concurso da Petrobras deve ...,A inscrição do concurso da Petrobras (Profissi...,O candidato deve realizar a inscrição para o c...,"A inscrição deve ser feita pela internet, no s...",A inscrição deve ser feita exclusivamente pela...,A inscrição deve ser feita somente pela intern...,A inscrição deveria ser realizada no site do C...,"De acordo com o item\n5.2\ndo edital, a inscri..."
3,4,Dados sobre as inscrições,Quais são as formas de pagamento da taxa de in...,"Com base no edital do concurso Petrobras, **a ...",## Forma de pagamento da taxa de inscrição — P...,"Com base no edital, a forma de pagamento da ta...",De acordo com o edital do concurso da Petrobra...,"Com base no edital, as formas de pagamento da ...",As formas de pagamento da taxa de inscrição do...,A taxa de inscrição deve ser paga por meio de ...,A taxa de inscrição deve ser paga por **boleto...,A taxa de inscrição deve ser paga **por meio d...,A taxa de inscrição deveria ser paga por meio ...,"De acordo com o item\n5.2.4\ndo edital, o paga..."
4,5,Dados sobre as inscrições,Qual é o prazo-limite de pagamento da inscriçã...,"Com base no edital da Petrobras, o **prazo-lim...",O prazo-limite de pagamento da taxa de inscriç...,A data-limite para pagamento da taxa de inscri...,De acordo com o **Anexo IV - Cronograma** do e...,O prazo-limite de pagamento da taxa de inscriç...,O prazo-limite para o pagamento da taxa de ins...,O prazo-limite para pagamento da inscrição foi...,O prazo-limite para pagamento da inscrição foi...,Hoje é 30/04/2026. A data limite para pagament...,O prazo-limite para pagamento da taxa de inscr...,De acordo com o\nAnexo IV (Cronograma)\ndo edi...
5,6,Dados sobre as inscrições,Quem pode pedir isenção da taxa de inscrição d...,"Com base no edital da Petrobras, **quem pode p...",## Quem pode pedir isenção da taxa de inscriçã...,Ótima pergunta! Veja abaixo quem pode solicita...,Aqui estão as informações completas sobre quem...,"Com base no edital, aqui está a resposta:\n\n-...",A isenção da taxa de inscrição do concurso da ...,Pode pedir isenção total da taxa apenas o(a) c...,Podem pedir isenção total da taxa de inscrição...,Podem pedir isenção total da taxa de inscrição...,Podiam solicitar isenção d

In [ ]:
from pathlib import Path
import pandas as pd

NB_DIR = Path('.').resolve()  # avaliacao_chats_comerciais/
DATA_DIR = NB_DIR / 'data'
DF_AVAL_PATH = NB_DIR.parent / 'avaliacao_llms' / 'df_avaliacoes.xlsx'
BATCHES_DIR = NB_DIR / 'batches'
BATCHES_DIR.mkdir(exist_ok=True)

In [5]:
from pathlib import Path
import pandas as pd

NB_DIR = Path('.').resolve()  # avaliacao_chats_comerciais/
DATA_DIR = NB_DIR / 'data'
DF_AVAL_PATH = NB_DIR.parent / 'avaliacao_llms' / 'df_avaliacoes.xlsx'
BATCHES_DIR = NB_DIR / 'batches'
BATCHES_DIR.mkdir(exist_ok=True)

def prepara_batches(concurso: str, chats: dict, n_batches: int = 5):
    """
    concurso: 'bndes' | 'cvm' | 'petrobras'
    chats: {'chatgpt': Path, 'claude_chat': Path} com xlsx contendo colunas id, resposta
    """
    df = pd.read_excel(DF_AVAL_PATH)
    sub = df[df['edital'] == concurso][['id', 'categoria', 'pergunta', 'modelo', 'resposta']]

    wide = (sub.pivot_table(index=['id', 'categoria', 'pergunta'],
                            columns='modelo', values='resposta', aggfunc='first')
              .reset_index())
    wide.columns = list(wide.columns[:3]) + [f'resposta_{c}' for c in wide.columns[3:]]

    for nome, path in chats.items():
        c = pd.read_excel(path)[['id', 'resposta']].rename(columns={'resposta': f'resposta_{nome}'})
        wide = wide.merge(c, on='id', how='left')

    wide = wide.sort_values('id').reset_index(drop=True)
    assert len(wide) == 50, f'esperado 50 perguntas, veio {len(wide)}'

    size = len(wide) // n_batches
    for i in range(n_batches):
        ini, fim = i * size, (i + 1) * size if i < n_batches - 1 else len(wide)
        out = BATCHES_DIR / f'{concurso}_batch_{i+1:02d}.xlsx'
        wide.iloc[ini:fim].to_excel(out, index=False)
        print(f'{out.name}: perguntas {wide.iloc[ini].id}–{wide.iloc[fim-1].id}')
    return wide

prepara_batches('cvm', {
    'chatgpt':     DATA_DIR / 'cvm_std_chatgpt_qa.xlsx',
    'claude_chat': DATA_DIR / 'cvm_std_claude_qa.xlsx',
})

cvm_batch_01.xlsx: perguntas 1–10
cvm_batch_02.xlsx: perguntas 11–20
cvm_batch_03.xlsx: perguntas 21–30
cvm_batch_04.xlsx: perguntas 31–40
cvm_batch_05.xlsx: perguntas 41–50


,id,categoria,pergunta,resposta_claude-haiku-4-5,resposta_claude-opus-4-7,resposta_claude-sonnet-4-6,resposta_deepseek-v4-flash,resposta_deepseek-v4-pro,resposta_gpt-4o-mini,resposta_gpt-5.4,resposta_gpt-5.4-mini,resposta_gpt-5.5,resposta_chatgpt,resposta_claude_chat
0,1,Dados sobre as inscrições,Qual o período de inscrição para o concurso CVM?,O período de inscrição para o concurso CVM é:\...,O período de inscrição para o concurso da CVM ...,Aqui estão as informações sobre o período de i...,O período de inscrição para o concurso da CVM ...,**Período de inscrição do concurso CVM (Edital...,As inscrições para o concurso da CVM estarão a...,O período de inscrição foi de 30 de janeiro de...,As inscrições para o concurso CVM estiveram ab...,Hoje é 30/04/2026. A inscrição para o concurso...,O período de inscrição para o concurso da CVM ...,"De acordo com o edital, as inscrições estavam ..."
1,2,Dados sobre as inscrições,Qual o valor da taxa de inscrição do concurso ...,O valor da taxa de inscrição do concurso CVM é...,A taxa de inscrição do concurso da CVM (Edital...,O valor da taxa de inscrição do concurso públi...,O valor da taxa de inscrição para o concurso d...,O valor da taxa de inscrição para o concurso d...,O valor da taxa de inscrição para o concurso d...,"A taxa de inscrição é de R$ 145,00.\n\nref: ed...","A taxa de inscrição é de R$ 145,00.\n\nref: ed...",A taxa de inscrição do concurso da CVM é de R$...,O valor da taxa de inscrição do concurso da CV...,"De acordo com o edital, o valor da taxa de ins..."
2,3,Dados sobre as inscrições,Em qual site o candidato deve fazer a inscriçã...,O candidato deve fazer a inscrição no seguinte...,A inscrição deve ser feita pelo site da FGV (b...,O candidato deve realizar a inscrição pelo seg...,O candidato deve realizar a inscrição por meio...,A inscrição para o concurso da CVM deve ser fe...,O candidato deve realizar a inscrição para o c...,"A inscrição deve ser feita no site da FGV, no ...",O candidato deve fazer a inscrição pela intern...,O candidato deve fazer a inscrição pelo site d...,A inscrição do concurso da CVM deveria ser rea...,"De acordo com o edital, as inscrições deveriam..."
3,4,Dados sobre as inscrições,Quais são as formas de pagamento da taxa de in...,As formas de pagamento da taxa de inscrição do...,## Formas de pagamento da taxa de inscrição — ...,Ótima pergunta! Aqui estão as formas de pagame...,As formas de pagamento da taxa de inscrição do...,"Com base no edital, as formas de pagamento da ...",As formas de pagamento da taxa de inscrição do...,A taxa de inscrição deve ser paga por meio da ...,"A taxa de inscrição deve ser paga por GRU, que...",A taxa de inscrição deve ser paga por **GRU (G...,A taxa de inscrição do concurso CVM poderia se...,"De acordo com o edital, o pagamento da taxa de..."
4,5,Dados sobre as inscrições,Qual é o prazo-limite de pagamento da inscriçã...,"Com base no edital, o **prazo-limite para paga...",O prazo-limite para pagamento da taxa de inscr...,O prazo-limite para pagamento da taxa de inscr...,"De acordo com o edital, o prazo-limite para pa...","Com base no edital, o prazo-limite para pagame...",O prazo-limite para o pagamento da taxa de ins...,O prazo-limite para pagar a taxa de inscrição ...,Hoje é 29/04/2026. O prazo-limite de pagamento...,Hoje é 30/04/2026. O prazo-limite para pagamen...,O prazo-limite para pagamento da taxa de inscr...,"De acordo com o edital, o prazo-limite para pa..."
5,6,Dados sobre as inscrições,Quem pode pedir isenção da taxa de inscrição d...,"Segundo o edital da CVM, **apenas dois grupos ...",Podem solicitar isenção da taxa de inscrição d...,"Com base no edital, **duas categorias de candi...",Podem pedir **isenção da taxa de inscrição** o...,"Com base no edital, aqui está a resposta compl...",A isenção da taxa de inscrição do concurso da ...,"Podem pedir isenção da taxa de inscrição, segu...",Podem pedir isenção da taxa de inscrição:\n\n-...,Pode pedir isenção da taxa de inscrição quem s...,Poderiam solicitar 

In [2]:
(NB_DIR / 'batches_eval').mkdir(exist_ok=True)

In [ ]:
def valida_batch(concurso: str, n: int):
    path = NB_DIR / 'batches_eval' / f'{concurso}_batch_{n:02d}_eval.xlsx'
    df = pd.read_excel(path)

    esperado = ['id', 'pergunta', 'modelo', 'resposta', 'score', 'justificativa']
    print(f'arquivo: {path.name}')
    print(f'shape:   {df.shape}  (esperado: 110, 6)')
    print(f'colunas: {list(df.columns)}')

    problemas = []
    if list(df.columns) != esperado:
        problemas.append(f'colunas fora de ordem/nome: {list(df.columns)}')
    if len(df) != 110:
        problemas.append(f'linhas != 110: {len(df)}')

    # score: aceita "1", "1.0", 1, 1.0 — mas normaliza pra float
    df['score'] = pd.to_numeric(df['score'].astype(str).str.replace(',', '.'), errors='coerce')
    fora = df[~df['score'].isin([0, 0.5, 1])]
    if len(fora):
        problemas.append(f'{len(fora)} scores fora de {{0, 0.5, 1}}')

    # cada pergunta deve ter exatamente 11 modelos
    cont = df.groupby('id')['modelo'].nunique()
    if not (cont == 11).all():
        problemas.append(f'perguntas com !=11 modelos: {cont[cont!=11].to_dict()}')

    # 10 perguntas distintas no batch
    if df['id'].nunique() != 10:
        problemas.append(f'ids distintos != 10: {df["id"].nunique()}')

    # justificativa nunca vazia
    vazias = df['justificativa'].isna().sum() + (df['justificativa'].astype(str).str.strip() == '').sum()
    if vazias:
        problemas.append(f'{vazias} justificativas vazias')

    # nomes de modelo consistentes entre batches: confere com o batch de entrada
    entrada = pd.read_excel(NB_DIR / 'batches' / f'{concurso}_batch_{n:02d}.xlsx')
    modelos_esperados = {c.replace('resposta_', '') for c in entrada.columns if c.startswith('resposta_')}
    modelos_vistos = set(df['modelo'].unique())
    if modelos_esperados != modelos_vistos:
        problemas.append(f'faltando: {modelos_esperados - modelos_vistos} | sobrando: {modelos_vistos - modelos_esperados}')

    if problemas:
        print('PROBLEMAS:')
        for p in problemas: print(f'  - {p}')
    else:
        print('OK')
        print('\ndistribuição de scores:')
        print(df['score'].value_counts().sort_index())
        print('\nmédia por modelo neste batch:')
        print(df.groupby('modelo')['score'].mean().sort_values(ascending=False).round(2))

    return df

# uso:
df2 = valida_batch('bndes', 2)

arquivo: bndes_batch_02_eval.xlsx
shape:   (110, 6)  (esperado: 110, 6)
colunas: ['id', 'pergunta', 'modelo', 'resposta', 'score', 'justificativa']
OK

distribuição de scores:
score
0.0     1
0.5    45
1.0    64
Name: count, dtype: int64

média por modelo neste batch:
modelo
gpt-5.4-mini         0.95
gpt-5.5              0.95
gpt-5.4              0.90
claude-opus-4-7      0.85
deepseek-v4-pro      0.80
claude-sonnet-4-6    0.75
chatgpt              0.75
gpt-4o-mini          0.75
deepseek-v4-flash    0.75
claude-haiku-4-5     0.70
claude_chat          0.50
Name: score, dtype: float64


In [6]:
def valida_batch(concurso: str, n: int):
    path = NB_DIR / 'batches_eval' / f'{concurso}_batch_{n:02d}_eval.xlsx'
    df = pd.read_excel(path)

    esperado = ['id', 'pergunta', 'modelo', 'resposta', 'score', 'justificativa']
    print(f'arquivo: {path.name}')
    print(f'shape:   {df.shape}  (esperado: 110, 6)')
    print(f'colunas: {list(df.columns)}')

    problemas = []
    if list(df.columns) != esperado:
        problemas.append(f'colunas fora de ordem/nome: {list(df.columns)}')
    if len(df) != 110:
        problemas.append(f'linhas != 110: {len(df)}')

    # score: aceita "1", "1.0", 1, 1.0 — mas normaliza pra float
    df['score'] = pd.to_numeric(df['score'].astype(str).str.replace(',', '.'), errors='coerce')
    fora = df[~df['score'].isin([0, 0.5, 1])]
    if len(fora):
        problemas.append(f'{len(fora)} scores fora de {{0, 0.5, 1}}')

    # cada pergunta deve ter exatamente 11 modelos
    cont = df.groupby('id')['modelo'].nunique()
    if not (cont == 11).all():
        problemas.append(f'perguntas com !=11 modelos: {cont[cont!=11].to_dict()}')

    # 10 perguntas distintas no batch
    if df['id'].nunique() != 10:
        problemas.append(f'ids distintos != 10: {df["id"].nunique()}')

    # justificativa nunca vazia
    vazias = df['justificativa'].isna().sum() + (df['justificativa'].astype(str).str.strip() == '').sum()
    if vazias:
        problemas.append(f'{vazias} justificativas vazias')

    # nomes de modelo consistentes entre batches: confere com o batch de entrada
    entrada = pd.read_excel(NB_DIR / 'batches' / f'{concurso}_batch_{n:02d}.xlsx')
    modelos_esperados = {c.replace('resposta_', '') for c in entrada.columns if c.startswith('resposta_')}
    modelos_vistos = set(df['modelo'].unique())
    if modelos_esperados != modelos_vistos:
        problemas.append(f'faltando: {modelos_esperados - modelos_vistos} | sobrando: {modelos_vistos - modelos_esperados}')

    if problemas:
        print('PROBLEMAS:')
        for p in problemas: print(f'  - {p}')
    else:
        print('OK')
        print('\ndistribuição de scores:')
        print(df['score'].value_counts().sort_index())
        print('\nmédia por modelo neste batch:')
        print(df.groupby('modelo')['score'].mean().sort_values(ascending=False).round(2))

    return df

# uso:
df2 = valida_batch('bndes', 4)

arquivo: bndes_batch_04_eval.xlsx
shape:   (110, 6)  (esperado: 110, 6)
colunas: ['id', 'pergunta', 'modelo', 'resposta', 'score', 'justificativa']
OK

distribuição de scores:
score
0.0     4
0.5    53
1.0    53
Name: count, dtype: int64

média por modelo neste batch:
modelo
gpt-5.5              1.00
gpt-5.4              0.95
gpt-5.4-mini         0.85
gpt-4o-mini          0.80
deepseek-v4-pro      0.75
claude-haiku-4-5     0.75
claude-opus-4-7      0.70
chatgpt              0.60
deepseek-v4-flash    0.55
claude-sonnet-4-6    0.50
claude_chat          0.50
Name: score, dtype: float64
